In [1]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import json
import math
import random
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    from peft import PeftModel
    HAS_PEFT = True
except Exception:
    HAS_PEFT = False

# ============================================================
# CONFIG
# ============================================================

BASE_MODEL = "microsoft/phi-3-mini-4k-instruct"
SFT_PATH = "/kaggle/input/datasets/adithyaled24b039/phi3-sft-folderr/phi3_sft_lora"

OUT_DIR = "/kaggle/working/phi3_mmlu_permutation_consistency"
CSV_DIR = os.path.join(OUT_DIR, "csv")
PLOTS_DIR = os.path.join(OUT_DIR, "plots")
REPORTS_DIR = os.path.join(OUT_DIR, "reports")
CACHE_DIR = os.path.join(OUT_DIR, "cache")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

SEED = 42
EVAL_SEED = 42

# Keep this moderate for Kaggle/T4.
SUBJECTS = [
    "abstract_algebra",
    "college_mathematics",
    "logical_fallacies",
    "formal_logic",
    "high_school_mathematics",
]

N_QUESTIONS_PER_SUBJECT = 8
N_PERMUTATIONS = 12
MAX_NEW_TOKENS = 16

# Prompt style: use the structured format you found helpful for MMLU.
USE_DELIMITER = True
DELIMITER_TEXT = "## Question:"

# Optional baseline comparison with the unpermuted original order.
RUN_ORIGINAL_ORDER = True

# Optional second prompt style for comparison.
RUN_PLAIN_PROMPT_BASELINE = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

TOKENIZER = None
MODEL = None

# ============================================================
# UTILS
# ============================================================

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def free_memory(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.synchronize()
        except Exception:
            pass

def save_df(df, path):
    ensure_dir(os.path.dirname(path))
    df.to_csv(path, index=False)

def save_json(obj, path):
    ensure_dir(os.path.dirname(path))
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def preview_text(s, max_len=220):
    s = str(s).replace("\n", " ")
    return s[:max_len] + ("..." if len(s) > max_len else "")

def to_jsonable(x):
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return json.dumps(str(x), ensure_ascii=False)

# ============================================================
# ADVANCED PLOTTING UTILS
# ============================================================

def bar_plot(labels, values, title, path, ylabel="Value"):
    plt.figure(figsize=(9, 4))
    plt.bar(labels, values, color='#4C72B0')
    plt.title(title)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def line_plot(x, ys, labels, title, path, xlabel="X", ylabel="Y"):
    plt.figure(figsize=(10, 5))
    for y, label in zip(ys, labels):
        plt.plot(x, y, marker="o", linewidth=1.5, label=label)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def heatmap(mat, xlabels, ylabels, title, path, xlabel="", ylabel="", cmap="viridis"):
    plt.figure(figsize=(max(8, len(xlabels) * 0.55), max(5, len(ylabels) * 0.35)))
    im = plt.imshow(mat, aspect="auto", cmap=cmap)
    plt.colorbar(im)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(range(len(xlabels)), xlabels, rotation=45, ha="right")
    plt.yticks(range(len(ylabels)), ylabels)
    
    # Annotate cells
    for i in range(len(ylabels)):
        for j in range(len(xlabels)):
            val = mat[i, j]
            color = "white" if val < (mat.max() / 2) else "black"
            plt.text(j, i, str(int(val)), ha="center", va="center", color=color)

    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def histogram(values, title, path, xlabel="Value", bins=10):
    plt.figure(figsize=(9, 4))
    plt.hist(values, bins=bins, color='#55A868', edgecolor='black', alpha=0.7)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def hexbin_plot(x, y, title, path, xlabel="X", ylabel="Y"):
    plt.figure(figsize=(8, 6))
    hb = plt.hexbin(x, y, gridsize=15, cmap='inferno', mincnt=1)
    cb = plt.colorbar(hb)
    cb.set_label('Question Count Density')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def radar_chart(categories, values_dict, title, path):
    N = len(categories)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    
    plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    
    plt.xticks(angles[:-1], categories, size=9)
    ax.set_rlabel_position(0)
    
    for label, values in values_dict.items():
        vals = list(values)
        vals += vals[:1]
        ax.plot(angles, vals, linewidth=2, linestyle='solid', label=label)
        ax.fill(angles, vals, alpha=0.25)
        
    plt.title(title, size=15, y=1.1)
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

# ============================================================
# PARSING UTILS
# ============================================================

def normalize_number(x):
    if x is None:
        return None
    try:
        return float(re.sub(r"[$,]", "", str(x)))
    except Exception:
        return None

def canonical_number_str(x):
    v = normalize_number(x)
    if v is None:
        return None
    if abs(v - round(v)) < 1e-6:
        return str(int(round(v)))
    return f"{v:.6f}".rstrip("0").rstrip(".")

def same_numeric(a, b, tol=1e-6):
    na = normalize_number(a)
    nb = normalize_number(b)
    if na is None or nb is None:
        return False
    return abs(na - nb) <= tol

def softmax_np(x):
    x = np.array(x, dtype=np.float64)
    x = x - np.max(x)
    ex = np.exp(x)
    return ex / (ex.sum() + 1e-12)

# ============================================================
# CACHE / SAMPLING
# ============================================================

def cached_indices(name, ds_len, n, seed=EVAL_SEED):
    ensure_dir(CACHE_DIR)
    n = min(n, ds_len)
    path = os.path.join(CACHE_DIR, f"{name}_n{n}_seed{seed}.json")
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    rng = np.random.RandomState(seed)
    idx = rng.choice(ds_len, size=n, replace=False).tolist()
    with open(path, "w") as f:
        json.dump(idx, f)
    return idx

def sample_dataset(ds, name, n, seed=EVAL_SEED):
    idx = cached_indices(name, len(ds), n, seed=seed)
    return ds.select(idx), idx

# ============================================================
# ANSWER EXTRACTION
# ============================================================

def extract_mcq(text, choices=None):
    if text is None:
        return None
    span = str(text)
    m = re.findall(r"<answer>(.*?)</answer>", span, flags=re.IGNORECASE | re.DOTALL)
    if m:
        span = m[-1]
    span_up = span.upper().strip()

    if span_up in ("A", "B", "C", "D"):
        return span_up

    patterns = [
        r"ANSWER\s*[:\-]?\s*\(?([ABCD])\)?",
        r"\b([ABCD])\b\s*$",
        r"<ANSWER>\s*\(?([ABCD])\)?",
        r"\bOPTION\s*([ABCD])\b",
    ]
    for pat in patterns:
        hits = re.findall(pat, span_up)
        if hits:
            return hits[-1].upper()

    letters = re.findall(r"\b([ABCD])\b", span_up)
    if letters:
        return letters[-1].upper()

    if choices is not None:
        span_low = span.lower()
        for i, choice in enumerate(choices):
            choice = str(choice).strip().lower()
            if choice and choice in span_low:
                return chr(65 + i)

    return None

def predicted_content_from_letter(letter, canonical_choices):
    if letter is None:
        return None
    letter = letter.upper().strip()
    if letter not in "ABCD":
        return None
    idx = ord(letter) - 65
    if idx < 0 or idx >= len(canonical_choices):
        return None
    return canonical_choices[idx]

# ============================================================
# MODEL LOADING
# ============================================================

def load_tokenizer():
    global TOKENIZER
    if TOKENIZER is None:
        src = SFT_PATH if (HAS_PEFT and os.path.exists(SFT_PATH)) else BASE_MODEL
        TOKENIZER = AutoTokenizer.from_pretrained(src)
        if TOKENIZER.pad_token is None:
            TOKENIZER.pad_token = TOKENIZER.eos_token
    return TOKENIZER

def load_model():
    global MODEL
    tok = load_tokenizer()

    kwargs = {"torch_dtype": DTYPE}
    try:
        kwargs["attn_implementation"] = "eager"
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)
    except Exception:
        kwargs.pop("attn_implementation", None)
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)

    model = model.to(DEVICE)
    model.eval()

    if HAS_PEFT and os.path.exists(SFT_PATH):
        try:
            model = PeftModel.from_pretrained(model, SFT_PATH).merge_and_unload().to(DEVICE)
            model.eval()
        except Exception:
            pass

    try:
        model.config.use_cache = False
    except Exception:
        pass

    MODEL = model
    return model, tok

# ============================================================
# PROMPTS
# ============================================================

def build_prompt(question, choices=None, delimiter=True, plain=False):
    header = DELIMITER_TEXT if delimiter else "Question:"
    if plain:
        header = "Question:"

    if choices is None:
        raise ValueError("choices must be provided for MMLU prompts.")

    options = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(choices)])
    prompt = (
        f"{header} {question}\n\n"
        f"{options}\n\n"
        "Answer ONLY with a single letter: A, B, C, or D."
    )
    return prompt

def build_permuted_prompt(question, choices, perm, delimiter=True, plain=False):
    perm_choices = [choices[i] for i in perm]
    return build_prompt(question, perm_choices, delimiter=delimiter, plain=plain)

# ============================================================
# DATA
# ============================================================

def load_subject_samples(subject, n, seed=EVAL_SEED):
    ds = load_dataset("cais/mmlu", subject, split="test")
    ds, idx = sample_dataset(ds, f"mmlu_{subject}", n, seed=seed)
    samples = []
    for i, s in enumerate(ds):
        samples.append({
            "question_id": f"{subject}_{i}",
            "subject": subject,
            "source_index": idx[i],
            "question": s["question"],
            "choices": list(s["choices"]),
            "gold_letter": chr(65 + int(s["answer"])),
            "gold_index": int(s["answer"]),
        })
    return samples

# ============================================================
# GENERATION / PARSING
# ============================================================

@torch.inference_mode()
def generate_once(prompt, max_new_tokens=MAX_NEW_TOKENS):
    inputs = TOKENIZER(prompt, return_tensors="pt").to(DEVICE)
    out = MODEL.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=TOKENIZER.eos_token_id,
        eos_token_id=TOKENIZER.eos_token_id,
    )
    full_output = TOKENIZER.decode(out[0], skip_special_tokens=True)
    prompt_len = inputs["input_ids"].shape[1]
    completion = TOKENIZER.decode(out[0][prompt_len:], skip_special_tokens=True)
    return full_output, completion

def canonical_from_permuted_letter(letter, perm):
    if letter is None:
        return None
    letter = letter.upper().strip()
    if letter not in "ABCD":
        return None
    idx = ord(letter) - 65
    if idx < 0 or idx >= len(perm):
        return None
    return chr(65 + perm[idx])

def letter_entropy(letter_counts):
    counts = np.array([letter_counts.get(k, 0) for k in ["A", "B", "C", "D"]], dtype=np.float64)
    p = counts / (counts.sum() + 1e-12)
    ent = -np.sum(np.where(p > 0, p * np.log2(p), 0.0))
    max_ent = math.log2(4)
    return float(ent), float(ent / max_ent if max_ent > 0 else 0.0)

# ============================================================
# EXPERIMENT RUNNER
# ============================================================

def run_question(sample, prompt_variant="anchored"):
    question = sample["question"]
    choices = sample["choices"]
    gold_letter = sample["gold_letter"]
    subject = sample["subject"]
    qid = sample["question_id"]

    delimiter = True
    plain = False
    if prompt_variant == "plain":
        delimiter = False
        plain = True

    original_prompt = build_prompt(question, choices, delimiter=delimiter, plain=plain)
    original_full, original_completion = generate_once(original_prompt)
    original_raw_letter = extract_mcq(original_completion, choices=choices)
    original_pred_letter = original_raw_letter
    original_pred_index = None if original_pred_letter is None else ord(original_pred_letter) - 65
    original_pred_content = predicted_content_from_letter(original_pred_letter, choices)
    original_correct = (original_pred_letter == gold_letter)

    rows = []
    perm_letters_raw = []
    perm_canonical_letters = []
    perm_content_preds = []
    perm_corrects = []

    if RUN_ORIGINAL_ORDER:
        rows.append({
            "question_id": qid,
            "subject": subject,
            "prompt_variant": prompt_variant,
            "mode": "original",
            "perm_id": -1,
            "question": question,
            "choices": to_jsonable(choices),
            "perm": to_jsonable(list(range(4))),
            "gold_letter": gold_letter,
            "gold_index": sample["gold_index"],
            "correct_position": sample["gold_index"],
            "chosen_position": original_pred_index if original_pred_index is not None else -1,
            "pred_raw_letter": original_pred_letter,
            "pred_canonical_letter": original_pred_letter,
            "pred_content": original_pred_content,
            "correct": bool(original_correct),
            "full_output": original_full,
            "completion": original_completion,
            "prompt": original_prompt,
            "prompt_tokens": len(TOKENIZER(original_prompt).input_ids),
        })

    for perm_id in range(N_PERMUTATIONS):
        perm = list(range(4))
        random.shuffle(perm)
        perm_choices = [choices[i] for i in perm]
        perm_prompt = build_permuted_prompt(question, choices, perm, delimiter=delimiter, plain=plain)
        
        full_output, completion = generate_once(perm_prompt)
        raw_letter = extract_mcq(completion, choices=perm_choices)
        canonical_letter = canonical_from_permuted_letter(raw_letter, perm)
        
        pred_content = None
        if canonical_letter is not None:
            pred_content = predicted_content_from_letter(canonical_letter, choices)

        correct = (canonical_letter == gold_letter)
        correct_position = perm.index(sample["gold_index"])
        chosen_position = ord(raw_letter) - 65 if (raw_letter and raw_letter in "ABCD") else -1

        perm_letters_raw.append(raw_letter)
        perm_canonical_letters.append(canonical_letter)
        perm_content_preds.append(pred_content)
        perm_corrects.append(correct)

        rows.append({
            "question_id": qid,
            "subject": subject,
            "prompt_variant": prompt_variant,
            "mode": "permuted",
            "perm_id": perm_id,
            "question": question,
            "choices": to_jsonable(choices),
            "perm": to_jsonable(perm),
            "perm_choices": to_jsonable(perm_choices),
            "gold_letter": gold_letter,
            "gold_index": sample["gold_index"],
            "correct_position": correct_position,
            "chosen_position": chosen_position,
            "pred_raw_letter": raw_letter,
            "pred_canonical_letter": canonical_letter,
            "pred_content": pred_content,
            "correct": bool(correct),
            "full_output": full_output,
            "completion": completion,
            "prompt": perm_prompt,
            "prompt_tokens": len(TOKENIZER(perm_prompt).input_ids),
        })

    canonical_counts = Counter([x for x in perm_canonical_letters if x is not None])
    raw_counts = Counter([x for x in perm_letters_raw if x is not None])
    content_counts = Counter([x for x in perm_content_preds if x is not None])

    canonical_mode, canonical_mode_count = canonical_counts.most_common(1)[0] if canonical_counts else (None, 0)
    raw_mode, raw_mode_count = raw_counts.most_common(1)[0] if raw_counts else (None, 0)
    content_mode, content_mode_count = content_counts.most_common(1)[0] if content_counts else (None, 0)

    consistency = canonical_mode_count / max(1, len(perm_canonical_letters))
    raw_bias_fraction = raw_mode_count / max(1, len(perm_letters_raw))
    content_stability = content_mode_count / max(1, len(perm_content_preds))
    perm_accuracy = float(np.mean(perm_corrects)) if perm_corrects else 0.0

    raw_letter_ent, raw_letter_rel_ent = letter_entropy(raw_counts)
    canon_letter_ent, canon_letter_rel_ent = letter_entropy(canonical_counts)

    summary = {
        "question_id": qid,
        "subject": subject,
        "prompt_variant": prompt_variant,
        "question": question,
        "choices": to_jsonable(choices),
        "gold_letter": gold_letter,
        "gold_index": sample["gold_index"],
        "original_pred_letter": original_pred_letter,
        "original_pred_content": original_pred_content,
        "original_correct": bool(original_correct),
        "perm_accuracy": perm_accuracy,
        "consistency_score": consistency,
        "canonical_mode_letter": canonical_mode,
        "raw_mode_letter": raw_mode,
        "content_mode": content_mode,
        "raw_bias_fraction": raw_bias_fraction,
        "content_stability": content_stability,
        "raw_letter_entropy": raw_letter_ent,
        "raw_letter_rel_entropy": raw_letter_rel_ent,
        "canonical_letter_entropy": canon_letter_ent,
        "canonical_letter_rel_entropy": canon_letter_rel_ent,
        "n_permutations": N_PERMUTATIONS,
    }

    return rows, summary, {
        "raw_counts": raw_counts,
        "canonical_counts": canonical_counts,
        "content_counts": content_counts,
        "perm_letters_raw": perm_letters_raw,
        "perm_canonical_letters": perm_canonical_letters,
        "perm_content_preds": perm_content_preds,
        "perm_corrects": perm_corrects,
        "original_pred_letter": original_pred_letter,
        "original_correct": original_correct,
        "gold_letter": gold_letter,
    }

# ============================================================
# AGGREGATION / METRICS
# ============================================================

def confusion_matrix_df(rows):
    labels = ["A", "B", "C", "D"]
    cm = pd.DataFrame(0, index=labels, columns=labels)
    for _, r in rows.iterrows():
        if r["mode"] != "permuted":
            continue
        gold = r["gold_letter"]
        pred = r["pred_canonical_letter"]
        if gold in labels and pred in labels:
            cm.loc[gold, pred] += 1
    return cm

def safe_mean(series):
    if len(series) == 0:
        return float("nan")
    return float(pd.Series(series).mean())

def make_plots_and_reports(sample_df, summary_df):
    overall = []
    for pv in sorted(summary_df["prompt_variant"].unique()):
        sdf = summary_df[summary_df["prompt_variant"] == pv]
        original_acc = safe_mean(sdf["original_correct"]) if "original_correct" in sdf.columns else float("nan")
        overall.append({
            "prompt_variant": pv,
            "n_questions": int(len(sdf)),
            "original_accuracy": original_acc,
            "permuted_accuracy": safe_mean(sdf["perm_accuracy"]),
            "consistency_score": safe_mean(sdf["consistency_score"]),
            "content_stability": safe_mean(sdf["content_stability"]),
            "raw_bias_fraction": safe_mean(sdf["raw_bias_fraction"]),
            "raw_letter_entropy": safe_mean(sdf["raw_letter_entropy"]),
            "canonical_letter_entropy": safe_mean(sdf["canonical_letter_entropy"]),
        })
    overall_df = pd.DataFrame(overall)
    save_df(overall_df, os.path.join(CSV_DIR, "overall_summary.csv"))

    subj_rows = []
    for pv in sorted(summary_df["prompt_variant"].unique()):
        for sub in sorted(summary_df["subject"].unique()):
            sdf = summary_df[(summary_df["prompt_variant"] == pv) & (summary_df["subject"] == sub)]
            if len(sdf) == 0:
                continue
            subj_rows.append({
                "prompt_variant": pv,
                "subject": sub,
                "n_questions": int(len(sdf)),
                "original_accuracy": safe_mean(sdf["original_correct"]),
                "permuted_accuracy": safe_mean(sdf["perm_accuracy"]),
                "consistency_score": safe_mean(sdf["consistency_score"]),
                "content_stability": safe_mean(sdf["content_stability"]),
                "raw_bias_fraction": safe_mean(sdf["raw_bias_fraction"]),
            })
    subject_df = pd.DataFrame(subj_rows)
    save_df(subject_df, os.path.join(CSV_DIR, "subject_summary.csv"))

    permdf = sample_df[sample_df["mode"] == "permuted"].copy()
    raw_counts_total = Counter(permdf["pred_raw_letter"].dropna().tolist())
    canon_counts_total = Counter(permdf["pred_canonical_letter"].dropna().tolist())

    letter_order = ["A", "B", "C", "D"]
    raw_counts = [raw_counts_total.get(k, 0) for k in letter_order]
    canon_counts = [canon_counts_total.get(k, 0) for k in letter_order]
    raw_total = sum(raw_counts)
    canon_total = sum(canon_counts)
    raw_fracs = [c / max(1, raw_total) for c in raw_counts]
    canon_fracs = [c / max(1, canon_total) for c in canon_counts]

    save_df(pd.DataFrame({"letter": letter_order, "count": raw_counts, "fraction": raw_fracs}), os.path.join(CSV_DIR, "raw_letter_distribution.csv"))
    save_df(pd.DataFrame({"letter": letter_order, "count": canon_counts, "fraction": canon_fracs}), os.path.join(CSV_DIR, "canonical_letter_distribution.csv"))

    bar_plot(letter_order, raw_counts, "Raw answer-letter distribution across permutations", os.path.join(PLOTS_DIR, "raw_letter_distribution.png"), ylabel="Count")
    bar_plot(letter_order, canon_counts, "Canonical answer-letter distribution across permutations", os.path.join(PLOTS_DIR, "canonical_letter_distribution.png"), ylabel="Count")

    for metric, title in [
        ("original_accuracy", "Original-order accuracy by prompt variant"),
        ("permuted_accuracy", "Permutation accuracy by prompt variant"),
        ("consistency_score", "Consistency score by prompt variant"),
        ("content_stability", "Content stability by prompt variant"),
        ("raw_bias_fraction", "Raw letter bias by prompt variant"),
    ]:
        bar_plot(
            overall_df["prompt_variant"].tolist(),
            overall_df[metric].tolist(),
            title,
            os.path.join(PLOTS_DIR, f"{metric}_by_prompt_variant.png"),
            ylabel="Metric Value"
        )

    # Subject-level bars (Robust Plotting via Pandas Pivot to avoid dimension mismatch)
    for metric in ["original_accuracy", "permuted_accuracy", "consistency_score", "content_stability", "raw_bias_fraction"]:
        if not subject_df.empty:
            pivot_df = subject_df.pivot(index="subject", columns="prompt_variant", values=metric)
            ax = pivot_df.plot(kind="bar", figsize=(10, 5), width=0.8, colormap="Set2")
            plt.title(f"Subject-level {metric}")
            plt.ylabel(metric)
            plt.xticks(rotation=30, ha="right")
            plt.tight_layout()
            plt.savefig(os.path.join(PLOTS_DIR, f"subject_{metric}.png"), dpi=180)
            plt.close()

    # Radar Chart for Multi-dimensional Subject Mapping
    if not subject_df.empty:
        subjects = sorted(subject_df["subject"].unique())
        for pv in overall_df["prompt_variant"].unique():
            values_dict = {}
            for metric in ["permuted_accuracy", "consistency_score", "content_stability"]:
                vals = []
                for sub in subjects:
                    val = subject_df[(subject_df["prompt_variant"] == pv) & (subject_df["subject"] == sub)][metric]
                    vals.append(float(val.iloc[0]) if not val.empty else 0.0)
                values_dict[metric] = vals
            
            radar_chart(
                subjects,
                values_dict,
                f"Subject-Level Metric Fingerprint ({pv})",
                os.path.join(PLOTS_DIR, f"radar_chart_{pv}.png")
            )

    histogram(
        summary_df["consistency_score"].tolist(),
        "Per-question consistency score distribution",
        os.path.join(PLOTS_DIR, "consistency_histogram.png"),
        xlabel="Consistency score",
        bins=10,
    )

    # High-Density Hexbin Plot for Accuracy vs Consistency
    valid_scatter = summary_df[["consistency_score", "perm_accuracy"]].dropna()
    if not valid_scatter.empty:
        hexbin_plot(
            valid_scatter["consistency_score"].tolist(),
            valid_scatter["perm_accuracy"].tolist(),
            "Density Map: Consistency vs Permutation Accuracy",
            os.path.join(PLOTS_DIR, "consistency_vs_accuracy_hexbin.png"),
            xlabel="Consistency score",
            ylabel="Permutation accuracy",
        )

    top_bias = summary_df.sort_values(["raw_bias_fraction", "consistency_score"], ascending=[False, False]).head(20)
    top_bias.to_csv(os.path.join(CSV_DIR, "top_bias_questions.csv"), index=False)

    top_unstable = summary_df.sort_values(["consistency_score", "perm_accuracy"], ascending=[True, True]).head(20)
    top_unstable.to_csv(os.path.join(CSV_DIR, "top_unstable_questions.csv"), index=False)

    cm = confusion_matrix_df(sample_df)
    save_df(cm.reset_index().rename(columns={"index": "gold_letter"}), os.path.join(CSV_DIR, "confusion_matrix.csv"))
    heatmap(
        cm.values,
        cm.columns.tolist(),
        cm.index.tolist(),
        "Gold letter vs canonical predicted letter",
        os.path.join(PLOTS_DIR, "confusion_matrix_heatmap.png"),
        xlabel="Predicted canonical letter",
        ylabel="Gold letter",
        cmap="Blues",
    )

    # Advanced Positional Bias Heatmap
    pos_labels = [0, 1, 2, 3]
    pos_cm = pd.DataFrame(0, index=pos_labels, columns=pos_labels)
    for _, r in permdf.iterrows():
        g_pos = r.get("correct_position", -1)
        c_pos = r.get("chosen_position", -1)
        if g_pos in pos_labels and c_pos in pos_labels:
            pos_cm.loc[g_pos, c_pos] += 1
            
    heatmap(
        pos_cm.values,
        ["A(0)", "B(1)", "C(2)", "D(3)"],
        ["A(0)", "B(1)", "C(2)", "D(3)"],
        "Positional Confusion: True Target vs Picked Position",
        os.path.join(PLOTS_DIR, "positional_bias_heatmap.png"),
        xlabel="Chosen Position (Raw Output)",
        ylabel="True Target Position",
        cmap="magma",
    )

    question_perm_df = summary_df[["question_id", "prompt_variant", "subject", "consistency_score", "perm_accuracy", "content_stability", "raw_bias_fraction"]].copy()
    save_df(question_perm_df, os.path.join(CSV_DIR, "question_summary.csv"))

    md = ["# MMLU permutation-consistency experiment\n"]
    md.append(f"- Model: `{BASE_MODEL}`")
    md.append(f"- Subjects: {', '.join(SUBJECTS)}")
    md.append(f"- Questions per subject: {N_QUESTIONS_PER_SUBJECT}")
    md.append(f"- Permutations per question: {N_PERMUTATIONS}")
    md.append(f"- Delimiter prompt: {USE_DELIMITER}")
    md.append(f"- Plain-prompt baseline: {RUN_PLAIN_PROMPT_BASELINE}")
    md.append("\n## Overall results\n")
    md.append("| Prompt variant | Original acc | Permuted acc | Consistency | Content stability | Raw bias |")
    md.append("|---|---:|---:|---:|---:|---:|")
    for _, r in overall_df.iterrows():
        md.append(
            f"| {r['prompt_variant']} | {r['original_accuracy']:.3f} | {r['permuted_accuracy']:.3f} | {r['consistency_score']:.3f} | {r['content_stability']:.3f} | {r['raw_bias_fraction']:.3f} |"
        )

    md.append("\n## Subject breakdown\n")
    md.append("| Prompt variant | Subject | Original acc | Permuted acc | Consistency | Content stability | Raw bias |")
    md.append("|---|---|---:|---:|---:|---:|---:|")
    for _, r in subject_df.iterrows():
        md.append(
            f"| {r['prompt_variant']} | {r['subject']} | {r['original_accuracy']:.3f} | {r['permuted_accuracy']:.3f} | {r['consistency_score']:.3f} | {r['content_stability']:.3f} | {r['raw_bias_fraction']:.3f} |"
        )

    md.append("\n## Interpretation hints\n")
    md.append("- High raw letter bias with lower canonical consistency suggests answer-position heuristics.")
    md.append("- High content stability with high permuted accuracy suggests genuine option-content reasoning.")
    md.append("- Low consistency but moderate original accuracy suggests order sensitivity.")
    md.append("- The confusion matrix shows whether certain gold letters are systematically favored.")
    md.append("- Positional Bias Heatmap (magma) reveals if your model falls back to picking 'C' when unsure, regardless of contents.")

    with open(os.path.join(REPORTS_DIR, "report.md"), "w") as f:
        f.write("\n".join(md))

    save_json({
        "overall": overall,
        "subjects": subject_df.to_dict(orient="records"),
        "config": {
            "BASE_MODEL": BASE_MODEL,
            "SFT_PATH": SFT_PATH,
            "SUBJECTS": SUBJECTS,
            "N_QUESTIONS_PER_SUBJECT": N_QUESTIONS_PER_SUBJECT,
            "N_PERMUTATIONS": N_PERMUTATIONS,
            "USE_DELIMITER": USE_DELIMITER,
            "DELIMITER_TEXT": DELIMITER_TEXT,
            "RUN_ORIGINAL_ORDER": RUN_ORIGINAL_ORDER,
            "RUN_PLAIN_PROMPT_BASELINE": RUN_PLAIN_PROMPT_BASELINE,
            "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
        }
    }, os.path.join(REPORTS_DIR, "summary.json"))

    return overall_df, subject_df, cm

# ============================================================
# MAIN
# ============================================================

def main():
    print("Loading model and tokenizer ...")
    load_model()
    print("Model loaded.")

    all_sample_rows = []
    all_summary_rows = []

    prompt_variants = ["anchored"]
    if RUN_PLAIN_PROMPT_BASELINE:
        prompt_variants.append("plain")

    for subject in SUBJECTS:
        print(f"\n=== Loading subject {subject} ===")
        samples = load_subject_samples(subject, N_QUESTIONS_PER_SUBJECT)

        for pv in prompt_variants:
            print(f"--- Prompt variant: {pv} ---")
            for i, sample in enumerate(samples):
                print(f"Question {i+1}/{len(samples)} | {preview_text(sample['question'], 90)}")
                rows, summary, _ = run_question(sample, prompt_variant=pv)
                all_sample_rows.extend(rows)
                all_summary_rows.append(summary)
                free_memory()

    sample_df = pd.DataFrame(all_sample_rows)
    summary_df = pd.DataFrame(all_summary_rows)

    save_df(sample_df, os.path.join(CSV_DIR, "permutation_results.csv"))
    save_df(summary_df, os.path.join(CSV_DIR, "question_summary_raw.csv"))

    overall_df, subject_df, cm = make_plots_and_reports(sample_df, summary_df)

    print("\n===== OVERALL SUMMARY =====")
    print(overall_df.to_string(index=False))
    print("\n===== SUBJECT SUMMARY =====")
    print(subject_df.to_string(index=False))
    print("\nSaved to:")
    print(f"- {OUT_DIR}")
    print(f"- CSVs: {CSV_DIR}")
    print(f"- Plots: {PLOTS_DIR}")
    print(f"- Reports: {REPORTS_DIR}")

if __name__ == "__main__":
    main()

Loading model and tokenizer ...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

high_school_mathematics/dev-00000-of-000(…):   0%|          | 0.00/4.50k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/270 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/29 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

--- Prompt variant: anchored ---
Question 1/8 | What is the ones digit of $1 \cdot 2 \cdot 3 \cdot 4 \cdot 5 \cdot 6 \cdot 7 \cdot 8 \cdot...
Question 2/8 | Evaluate the sum\[\frac{1}{2^1} + \frac{2}{2^2} + \frac{3}{2^3} + \cdots + \frac{k}{2^k} +...
Question 3/8 | A virus is spreading throughout the population of a town, and the number of people who hav...
Question 4/8 | Find the length of the curve y = ln x between the points where y = 1/2 and y = 1.
Question 5/8 | Let $p$, $q$, and $r$ be constants. One solution to the equation $(x-p)(x-q) = (r-p)(r-q)$...
Question 6/8 | The shape of the sign outside Bob's Burger Barn is a regular octagon. How many degrees are...
Question 7/8 | The $n^{\text{th}}$ term of a certain geometric series is given by $a\cdot r^{n-1}$, where...
Question 8/8 | A box contains 4 white balls and 4 black balls. I draw them out of the box, one at a time....

===== OVERALL SUMMARY =====
prompt_variant  n_questions  original_accuracy  permuted_accuracy  consistency